# DATALAB MEF Maroc - UC-S1 - Etape 3
## Benchlearning, simulation des trajectoires et livraison finale

Notebook maitre : orchestre la lecture des sorties des parties 1 a 5
(donneurs/poids, projections tendancielles, backtest de la combinaison,
scenarios, Monte Carlo) et presente les resultats cles avec commentaire
economique.

Le calcul complet est effectue par les scripts
`run_etape3_part1_donors_weights.py` a `run_etape3_part5_montecarlo.py`
(executables independamment depuis un terminal, dans cet ordre). Ce
notebook charge et commente leurs sorties -- il ne relance pas les
projections completes (plusieurs minutes de calcul) pour rester
utilisable de maniere interactive.

Convention de statut : **implemente**, **execute**, **evalue**, **valide**,
**non concluant**, **bloque**.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import json
import pandas as pd

from src import io_utils

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

cfg2 = io_utils.load_config("configs/config_etape2.yaml")
cfg3 = io_utils.load_config("configs/config_etape3.yaml")
tables_dir = Path(cfg3["paths"]["outputs_tables_dir"])
print("Dossier des tables etape 3 :", tables_dir.resolve())


C:\Users\MR KITOHOU\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Dossier des tables etape 3 : C:\Users\MR KITOHOU\Downloads\jarvis-starter-kit\mef_maroc_ucs1\outputs\etape3\tables


## 1. Validation de la configuration

In [2]:
from src.etape3 import config_check
checks = config_check.validate_config_etape3(cfg3, cfg2)
for c in checks:
    print("OK -", c)


OK - perimetre historique coherent avec l'etape 2
OK - projection_year_start contigu a year_end
OK - horizon de projection <= plafond configure
OK - Maroc exclu de ses propres donneurs
OK - horizons testes + extrapolation couvrent exactement 1..horizon_max
OK - seuil 0.55 non applique a S_POC (garde-fou respecte)
OK - scenarios politiques : poids >= 0 et somme = 1 (Eq. 3.14)
OK - alpha_s dans [0,1] (Eq. 3.15)
OK - ajustement dynamique (hors cadrage) desactive par defaut
OK - n_draws_full >= 10000 (cadrage Ch.6.4)


**Statut : execute.** 10 verifications bloquantes passees (perimetre
historique, horizons, garde-fous sur les poids et le seuil S_POC,
ajustement dynamique desactive, Monte Carlo >= 10000 tirages). Toute
incoherence leve une exception explicite plutot qu'un avertissement muet.

## 2. Donneurs : score S_POC et portefeuille

In [3]:
score_poc = pd.read_csv(tables_dir / "e3_01_score_poc.csv")
score_poc


,pays,S2_structure,S3_ouverture,S6_completude,S7_partiel,n_composantes_valides,S_POC
0,CHL,0.919172,0.939951,1.000000,0.752935,4,0.903014
1,TUR,0.955484,0.623096,0.938235,1.000000,4,0.879204
2,URY,0.898428,0.587083,0.972549,0.881209,4,0.834817
3,PRT,0.870688,0.846007,0.988235,0.631362,4,0.834073
4,CRI,0.907556,0.742981,0.995098,0.661129,4,0.826691
5,DNK,0.868925,0.324396,1.000000,0.928705,4,0.780507
6,VNM,0.895231,0.000000,0.921569,0.949767,4,0.691642
7,MYS,0.878719,0.000000,1.000000,0.000000,4,0.469680


**Score S_POC (exploratoire).** Construit sur 4 composantes
disponibles (S2 structure productive, S3 ouverture commerciale, S6
completude, S7 partiel -- proxy sur la seule progression de la FBCF). S1
(PIB/habitant PPA), S4 (contraintes en ressources) et S5 (qualite
institutionnelle WGI) sont absents des donnees disponibles : **S_POC n'est
jamais compare au seuil 0.55 du cadrage**, qui s'applique au score complet
a 7 criteres.

In [4]:
portfolios = pd.read_csv(tables_dir / "e3_03_portefeuilles_compares.csv")
portfolios.head(10)


,portefeuille,score_moyen,redondance,objectif
0,"('CHL', 'PRT', 'VNM')",0.809576,0.317493,0.650830
1,"('CHL', 'URY', 'VNM')",0.809825,0.335730,0.641960
2,"('CHL', 'DNK', 'VNM')",0.791721,0.302890,0.640276
3,"('CHL', 'TUR', 'DNK')",0.854242,0.429580,0.639451
4,"('CHL', 'CRI', 'VNM')",0.807116,0.345200,0.634516
5,"('CHL', 'TUR', 'CRI')",0.869636,0.477810,0.630732
6,"('CHL', 'TUR', 'PRT')",0.872097,0.483932,0.630131
7,"('CHL', 'TUR', 'URY')",0.872345,0.487667,0.628512
8,"('CHL', 'PRT', 'CRI')",0.854593,0.453577,0.627804
9,"('TUR', 'PRT', 'CRI')",0.846656,0.438666,0.627323


**Portefeuille initial vs alternatif (Eq. 3.12-3.13).** Le portefeuille
initial du cadrage (Viet Nam, Costa Rica, Danemark) est compare
exhaustivement aux 56 combinaisons possibles de taille 3 parmi les 8
candidats. L'alternative legerement superieure selon l'objectif
qualite-diversite ne remet pas en cause le choix narratif du cadrage --
c'est une sensibilite, pas un verdict.

## 3. Poids de combinaison : statistiques vs politiques

In [5]:
weights_common = pd.read_csv(tables_dir / "e3_04_poids_statistiques_communs.csv")
weights_common


,portefeuille,pays,poids_statistique_commun,n_cibles_utilisees
0,initial_cadrage,VNM,0.162187,7
1,initial_cadrage,CRI,0.410964,7
2,initial_cadrage,DNK,0.426849,7
3,alternatif_8pays,VNM,0.054920,7
4,alternatif_8pays,CRI,0.000000,7
5,alternatif_8pays,DNK,0.032943,7
6,alternatif_8pays,CHL,0.447584,7
7,alternatif_8pays,MYS,0.018032,7
8,alternatif_8pays,PRT,0.308647,7
9,alternatif_8pays,TUR,0.000000,7


**Poids statistiques (controle synthetique regularise).** Pour le
portefeuille initial, le Danemark et le Costa Rica dominent la combinaison
qui reproduit le mieux l'historique marocain -- le Viet Nam, malgre son
role narratif central dans le cadrage, pese moins dans la reference
statistique. Les poids **politiques** (scenarios mono-pays) sont
strictement separes et jamais utilises pour la prevision centrale (garde
-fou du cadrage Ch.3.5.1) : voir `e3_07_poids_politiques.csv`.

In [6]:
coherence = pd.read_csv(tables_dir / "e3_06_coherence_poids_par_cible.csv")
print("Correlation moyenne entre vecteurs de poids par cible :", coherence["correlation"].mean().round(3))
coherence


Correlation moyenne entre vecteurs de poids par cible : -0.025


,cible_1,cible_2,distance_L1,correlation
0,CROISSANCE_PIB,INFLATION_CPI,0.818007,0.926462
1,CROISSANCE_PIB,SOBG,1.270955,-0.947173
2,CROISSANCE_PIB,DETTE_PUBLIQUE,1.182864,-0.462167
3,CROISSANCE_PIB,BALANCE_COURANTE,1.502253,-0.691739
4,CROISSANCE_PIB,TACH,0.758951,0.160139
5,CROISSANCE_PIB,FBCF,0.960205,-0.438858
6,INFLATION_CPI,SOBG,2.000000,-0.756804
7,INFLATION_CPI,DETTE_PUBLIQUE,1.410558,-0.094403
8,INFLATION_CPI,BALANCE_COURANTE,1.729947,-0.369063
9,INFLATION_CPI,TACH,0.986645,0.519893


**Coherence poids communs vs poids par cible.** La correlation moyenne
proche de zero indique que les poids optimaux different sensiblement d'une
cible a l'autre : le portefeuille "commun" est une simplification
pratique, pas un resume fidele d'un ajustement cible par cible. Limite
documentee.

## 4. Backtest honnete de la reference de benchlearning (h=1-5)

In [7]:
eval_bench = pd.read_csv(tables_dir / "e3_21_evaluation_benchlearning.csv")
finale_vs_rw = eval_bench[(eval_bench.bloc == "finale") & (eval_bench.comparaison == "benchlearning_vs_random_walk")]
finale_vs_rw[["cible", "horizon", "n_previsions", "rmse_modele", "rmse_baseline", "gain_rmse_vs_baseline_pct", "rmse_relative_modele", "rmse_relative_baseline"]]


,cible,horizon,n_previsions,rmse_modele,rmse_baseline,gain_rmse_vs_baseline_pct,rmse_relative_modele,rmse_relative_baseline
2,CROISSANCE_PIB,1,4,3.272179,8.347698,60.801419,NaN,NaN
6,CROISSANCE_PIB,2,4,2.966836,5.759957,48.492038,NaN,NaN
10,CROISSANCE_PIB,3,4,2.878856,6.392486,54.964999,NaN,NaN
14,CROISSANCE_PIB,4,4,2.900830,5.747183,49.526056,NaN,NaN
18,CROISSANCE_PIB,5,4,3.033229,4.182251,27.473768,NaN,NaN
22,INFLATION_CPI,1,4,2.664473,3.690888,27.809432,NaN,NaN
26,INFLATION_CPI,2,4,3.267332,4.763920,31.415040,NaN,NaN
30,INFLATION_CPI,3,4,2.607075,4.174453,37.546905,NaN,NaN
34,INFLATION_CPI,4,4,3.593745,3.793018,5.253661,NaN,NaN
38,INFLATION_CPI,5,4,2.601467,3.665303,29.024506,NaN,NaN


**Resultat central, non lisse.** La reference statistique de
benchlearning (VNM/CRI/DNK ponderes, poids reestimes a chaque origine, sans
aucune fuite des realisations futures des donneurs) **bat la marche
aleatoire uniquement sur CROISSANCE_PIB (27 a 61% de gain) et
INFLATION_CPI (5 a 38%)**. Sur SOBG, DETTE_PUBLIQUE, BALANCE_COURANTE et
TACH, elle fait **nettement moins bien** qu'une simple persistance (RMSE
relative 2 a 3 fois superieure a la baseline pour BALANCE_COURANTE ; gain
negatif de -95% a -240% pour TACH). Ce constat n'a pas ete corrige apres
coup : il indique que le benchlearning international apporte une valeur
reelle pour les cibles cycliques mais pas pour les cibles structurelles,
avec cette methode et ces donnees.

## 5. Combinaison (Eq. 3.14) et trajectoires scenarisees (Eq. 3.15), 2025-2034

In [8]:
scenarios = pd.read_csv(tables_dir / "e3_32_trajectoires_scenarisees.csv")
scenarios[(scenarios.cible == "CROISSANCE_PIB") & (scenarios.horizon.isin([1, 5, 10]))][
    ["horizon", "annee_projetee", "scenario", "alpha", "X_tend_MAR", "X_bench_comb", "X_scenario", "horizon_teste_backtest"]
].sort_values(["horizon", "alpha"])


,horizon,annee_projetee,scenario,alpha,X_tend_MAR,X_bench_comb,X_scenario,horizon_teste_backtest
0,1,2025,tendance_nationale_seule,0.00,5.049983,3.248918,5.049983,True
70,1,2025,convergence_faible,0.25,5.049983,3.248918,4.599717,True
140,1,2025,convergence_moderee,0.50,5.049983,3.248918,4.149451,True
210,1,2025,convergence_forte,0.75,5.049983,3.248918,3.699184,True
280,1,2025,benchmark_pur,1.00,5.049983,3.248918,3.248918,True
4,5,2029,tendance_nationale_seule,0.00,3.746100,3.259898,3.746100,True
74,5,2029,convergence_faible,0.25,3.746100,3.259898,3.624549,True
144,5,2029,convergence_moderee,0.50,3.746100,3.259898,3.502999,True
214,5,2029,convergence_forte,0.75,3.746100,3.259898,3.381449,True
284,5,2029,benchmark_pur,1.00,3.746100,3.259898,3.259898,True


**Lecture.** Pour CROISSANCE_PIB, la tendance nationale (issue du
modele valide en etape 2) est plus dynamique que le benchmark combine des
2025 : un alpha eleve represente ici un ralentissement volontaire vers un
rythme de croissance de reference, pas un gain automatique. Les horizons
6-10 (2030-2034, `horizon_teste_backtest=False`) sont des extrapolations
non testees en backtest -- a traiter avec prudence accrue.

In [9]:
checks = pd.read_csv(tables_dir / "e3_34_verifications_coherence.csv")
checks


,test,reussi,ecart_max
0,alpha=0 => X_s = X_tend_MAR,True,0.0
1,alpha=1 => X_s = X_bench_comb,True,0.0
2,portefeuille a un seul donneur (VNM) => benchm...,True,0.0
3,poids ne sommant pas a 1 => rejet,True,0.0
4,poids negatif => rejet,True,0.0


**Verifications de coherence : 5/5 reussies.** alpha=0 reproduit
exactement la tendance nationale ; alpha=1 reproduit exactement le
benchmark combine ; un portefeuille a un seul donneur reproduit exactement
sa trajectoire ; poids ne sommant pas a 1 ou negatif : rejetes.

## 6. Simulation Monte Carlo jointe (10 000 tirages)

In [10]:
bands = pd.read_csv(tables_dir / "e3_43_bandes_incertitude.csv")
sample = bands[(bands.cible == "CROISSANCE_PIB") & (bands.scenario == "convergence_moderee") & (bands.distribution == "gaussienne")]
sample[["horizon", "annee_projetee", "mediane", "borne_basse_68", "borne_haute_68", "borne_basse_95", "borne_haute_95", "horizon_teste_backtest"]]


,horizon,annee_projetee,mediane,borne_basse_68,borne_haute_68,borne_basse_95,borne_haute_95,horizon_teste_backtest
420,1,2025,4.169914,0.862623,7.432585,-2.376350,10.472962,True
434,2,2026,3.417042,-1.260173,8.031156,-5.840773,12.330898,True
448,3,2027,3.453501,-2.274894,9.104613,-7.884961,14.370700,True
462,4,2028,3.443213,-3.171368,9.968555,-9.649315,16.049309,True
476,5,2029,3.548756,-3.846570,10.844310,-11.089135,17.642799,True
490,6,2030,3.384222,-4.716952,11.376102,-12.650784,18.823473,False
504,7,2031,3.933640,-4.816628,12.565857,-13.386146,20.609937,False
518,8,2032,3.385549,-5.968880,12.613777,-15.130081,21.213261,False
532,9,2033,2.838816,-7.083054,12.626830,-16.799975,21.747960,False
546,10,2034,3.232028,-7.226542,13.549501,-17.469076,23.164016,False


In [11]:
conv = pd.read_csv(tables_dir / "e3_42_controle_convergence.csv")
pivot = conv.pivot_table(index="cible", columns="n_tirages", values="q97_5")
pivot


n_tirages,1000,5000,10000,20000
cible,,,,
BALANCE_COURANTE,5.892264,5.916943,5.848512,5.862447
CROISSANCE_PIB,6.237578,6.259222,6.323511,6.410527
DETTE_PUBLIQUE,8.847478,9.327181,9.467979,9.367692
FBCF,3.788156,4.053205,4.121987,4.277404
INFLATION_CPI,3.894694,3.896343,3.814787,3.774479
SOBG,3.833790,3.850743,3.852963,3.811314
TACH,1.728861,1.655389,1.651180,1.640681


**Controle de convergence.** Ecart relatif maximal du quantile 97.5%
entre 1000 et 20000 tirages : environ 11%, porte principalement par FBCF
(cible la plus bruitee des 7 dans la matrice de residus). Les autres
cibles stabilisent a moins de 5% des 10000 tirages. Convergence jugee
acceptable mais pas parfaite -- rapportee telle quelle.

**Bandes livrees comme "bandes conditionnelles aux hypotheses"** (choix de
distribution des chocs, covariance historique, scenario alpha), pas comme
une couverture statistiquement verifiee : le cadrage demande une tolerance
de +/-5 points sur la couverture empirique, non verifiable finement avec
seulement 4 realisations par horizon (bloc final de l'etape 2).

## 7. Bilan de l'etape 3 et du projet complet

Voir `docs/ETAPE3_BENCHLEARNING_METHODE.md`, `docs/ETAPE3_RESULTATS_SIMULATION.md`
et `docs/SYNTHESE_METIER_FINALE.md` pour la synthese complete redigee, et
`docs/MATRICE_CONFORMITE.md` pour chaque decision documentee sur les 3
etapes du projet UC-S1.